# 05 — Decide

Summarise the simulation results and recommend accept / accept with caveats / reject / refine.

**Input:** `04-simulation.jsonl`, `03-proposed-change.md`, `01-worst-briefs.jsonl` (for original per-criterion scores).

**Output:** `05-decision.md` — advisory recommendation with score-delta summary.

No LLM call. Deterministic summary only.

See [paper-brief-improvement.md](../../docs/specs/paper-brief-improvement.md) step 5.

In [1]:
# Improvement run folder under data/paper_brief_improvement/.
# Leave empty to use the latest folder that has 04-simulation.jsonl.
RUN_ID = ""

In [2]:
from __future__ import annotations

import json
import re
import statistics
from pathlib import Path

from IPython.display import Markdown, display


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
IMPROVEMENT_PARENT = REPO_ROOT / "data" / "paper_brief_improvement"

print(f"repo root: {REPO_ROOT}")
print(f"improvement parent: {IMPROVEMENT_PARENT}")

repo root: /workspace
improvement parent: /workspace/data/paper_brief_improvement


In [3]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")
CRITERIA = ("faithfulness", "completeness", "conciseness", "topic_agnostic")


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def resolve_run_dir(run_id: str) -> Path:
    if run_id:
        d = IMPROVEMENT_PARENT / run_id
        if not (d / "04-simulation.jsonl").is_file():
            raise FileNotFoundError(f"No 04-simulation.jsonl in {d}")
        if not (d / "03-proposed-change.md").is_file():
            raise FileNotFoundError(f"No 03-proposed-change.md in {d}")
        return d
    candidates = sorted(
        (
            p.parent
            for p in IMPROVEMENT_PARENT.glob("*/04-simulation.jsonl")
            if _RUN_ID_PATTERN.match(p.parent.name)
        ),
        key=lambda d: d.name,
    )
    if not candidates:
        raise FileNotFoundError(
            "No improvement runs with 04-simulation.jsonl found under "
            + str(IMPROVEMENT_PARENT)
        )
    return candidates[-1]


def criterion_score(evaluation: dict | None, name: str) -> int | None:
    if not isinstance(evaluation, dict):
        return None
    entry = evaluation.get(name, {})
    if not isinstance(entry, dict):
        return None
    score = entry.get("score")
    if isinstance(score, bool) or not isinstance(score, int):
        return None
    return score


def recommend(
    mean_delta: float,
    improved: int,
    degraded: int,
) -> tuple[str, str]:
    """Return (label, rationale) per the improvement-workflow rules."""
    if mean_delta > 0 and degraded == 0:
        return (
            "accept",
            "Mean delta is positive and no simulated paper degraded.",
        )
    if mean_delta > 0 and degraded > 0:
        if improved > 0 and degraded >= improved:
            return (
                "refine",
                "Mean delta is positive, but as many or more papers degraded "
                "as improved. A narrower change is likely safer.",
            )
        return (
            "accept with caveats",
            "Mean delta is positive, but at least one paper degraded.",
        )
    if improved > 0:
        return (
            "refine",
            "Mean delta is not positive, yet some papers improved. "
            "Narrow or revise the proposal before accepting.",
        )
    return (
        "reject",
        "Mean delta is not positive and no simulated paper improved.",
    )

In [4]:
run_dir = resolve_run_dir(RUN_ID)
print(f"run dir: {run_dir.relative_to(REPO_ROOT)}")

simulation_rows = load_jsonl(run_dir / "04-simulation.jsonl")
proposal_md = (run_dir / "03-proposed-change.md").read_text(encoding="utf-8")

worst_by_doi: dict[str, dict] = {}
worst_path = run_dir / "01-worst-briefs.jsonl"
if worst_path.is_file():
    for row in load_jsonl(worst_path):
        doi = row.get("doi")
        if isinstance(doi, str) and doi:
            worst_by_doi[doi] = row

successful = [r for r in simulation_rows if isinstance(r.get("delta"), (int, float))]
errors = [r for r in simulation_rows if "error" in r and "delta" not in r]

print(f"simulation rows: {len(simulation_rows)}")
print(f"successful: {len(successful)}")
print(f"errors: {len(errors)}")

if not successful:
    raise RuntimeError(
        "No successful simulation rows with a delta. "
        "Run notebook 04 (simulate change) first."
    )

run dir: data/paper_brief_improvement/20260818T221210Z_gemma4-e4b
simulation rows: 5
successful: 5
errors: 0


In [5]:
deltas = [float(r["delta"]) for r in successful]
improved = sum(1 for d in deltas if d > 0)
degraded = sum(1 for d in deltas if d < 0)
unchanged = sum(1 for d in deltas if d == 0)
mean_delta = statistics.mean(deltas)
median_delta = statistics.median(deltas)

criterion_deltas: dict[str, list[float]] = {c: [] for c in CRITERIA}
for row in successful:
    doi = row["doi"]
    original_eval = (worst_by_doi.get(doi) or {}).get("evaluation")
    new_eval = row.get("new_evaluation")
    for c in CRITERIA:
        old_s = criterion_score(original_eval, c)
        new_s = criterion_score(new_eval, c)
        if old_s is not None and new_s is not None:
            criterion_deltas[c].append(float(new_s - old_s))

label, rationale = recommend(mean_delta, improved, degraded)

print(f"mean delta: {mean_delta:+.2f}")
print(f"median delta: {median_delta:+.2f}")
print(f"improved / degraded / unchanged: {improved} / {degraded} / {unchanged}")
print(f"recommendation: {label}")

mean delta: +0.55
median delta: +1.00
improved / degraded / unchanged: 3 / 1 / 1
recommendation: accept with caveats


In [6]:
lines: list[str] = [
    "# Decision",
    "",
    f"**Recommendation:** {label}",
    "",
    rationale,
    "",
    "This recommendation is advisory. A developer or agent must apply "
    "(or reject) the change outside this workflow.",
    "",
    "## Simulation results",
    "",
    f"- Papers simulated: {len(successful)}",
    f"- Errors skipped: {len(errors)}",
    f"- Mean delta: {mean_delta:+.2f}",
    f"- Median delta: {median_delta:+.2f}",
    f"- Improved (delta > 0): {improved}",
    f"- Degraded (delta < 0): {degraded}",
    f"- Unchanged (delta == 0): {unchanged}",
    "",
    "### Per-paper scores",
    "",
    "| DOI | Original | New | Delta |",
    "| --- | ---: | ---: | ---: |",
]

for row in successful:
    lines.append(
        f"| `{row['doi']}` | {float(row['original_score']):.2f} | "
        f"{float(row['new_score']):.2f} | {float(row['delta']):+.2f} |"
    )

lines.extend(["", "### Mean criterion delta (new − original)", ""])
for c in CRITERIA:
    values = criterion_deltas[c]
    if values:
        lines.append(f"- **{c}**: {statistics.mean(values):+.2f}")
    else:
        lines.append(f"- **{c}**: –")

if errors:
    lines.extend(["", "### Simulation errors", ""])
    for row in errors:
        lines.append(f"- `{row.get('doi', '?')}`: {row.get('error', 'unknown')}")

lines.extend([
    "",
    "## Proposed change",
    "",
    "Source: `03-proposed-change.md`",
    "",
    proposal_md.strip(),
    "",
])

decision_md = "\n".join(lines) + "\n"
decision_path = run_dir / "05-decision.md"
decision_path.write_text(decision_md, encoding="utf-8")
print(f"wrote {decision_path.relative_to(REPO_ROOT)}")

display(Markdown(decision_md))

wrote data/paper_brief_improvement/20260818T221210Z_gemma4-e4b/05-decision.md


# Decision

**Recommendation:** accept with caveats

Mean delta is positive, but at least one paper degraded.

This recommendation is advisory. A developer or agent must apply (or reject) the change outside this workflow.

## Simulation results

- Papers simulated: 5
- Errors skipped: 0
- Mean delta: +0.55
- Median delta: +1.00
- Improved (delta > 0): 3
- Degraded (delta < 0): 1
- Unchanged (delta == 0): 1

### Per-paper scores

| DOI | Original | New | Delta |
| --- | ---: | ---: | ---: |
| `10.1093/JME/TJAG095` | 4.00 | 5.00 | +1.00 |
| `10.3390/VACCINES14060499` | 4.00 | 5.00 | +1.00 |
| `10.64898/2026.05.10.722846` | 4.00 | 5.00 | +1.00 |
| `10.1016/J.APSB.2026.02.021` | 4.25 | 4.00 | -0.25 |
| `10.1038/S41467-026-73251-5` | 4.25 | 4.25 | +0.00 |

### Mean criterion delta (new − original)

- **faithfulness**: +1.80
- **completeness**: -0.40
- **conciseness**: +0.40
- **topic_agnostic**: +0.40

## Proposed change

Source: `03-proposed-change.md`

## Rationale
The current instructions for `key_findings` are too passive; they merely ask the model to list "primary metrics or results." However, high-quality scientific briefs (as demonstrated by the diagnosis summary) do not just list data points; they synthesize a finding *and* its immediate implication. By modifying this section's prompt, we force the model to act as an interpreter of the raw data, ensuring that every listed key finding is presented with enough context or significance statement derived from the text (e.g., "X was found in Y areas, suggesting Z risk"). This elevates the brief from a mere summary of results to a true synthesis of scientific impact.

## Proposed change
Modify the description for `key_findings` within the `# Output fields` section.

**Old Text:**
```markdown
### `key_findings` (required)

A short list (typically two or three items) of primary metrics or results. Do not dump every table.
```

**New Text:**
```markdown
### `key_findings` (required)

A short, synthesized list (typically two to four items) detailing the most impactful primary metrics or results. Each finding must state the result and its immediate significance or implication as described by the authors in the text. Do not simply list data points; synthesize the key takeaway.
```

## Expected effect on G-Eval criteria
**faithfulness:** Improve. By requiring the model to explicitly link a finding to its *significance* (which must still be grounded in the source text), we force a deeper, more accurate understanding of the paper's core message, reducing the risk of listing isolated data points without context.

**completeness:** Improve. The change guides the model toward capturing the full scope of an impactful finding—the result *and* its meaning—making the brief feel richer and more complete in conveying scientific insight.

**conciseness:** No effect. While the instruction is slightly longer, it improves information density by ensuring that every bullet point carries both factual weight and interpretive value, maintaining conciseness without sacrificing depth.

**topic_agnostic:** No effect. The change relates purely to the structural requirement of synthesis for a field, not the nature or topic of the science being summarized.

